# Revision figures

This small notebook is used to construct the figures that have been suggested by reviewers.

## 1. ROC curves

In [8]:
import pandas as pd
from sklearn.metrics import roc_auc_score

df = pd.read_csv("test_predictions_before_calibration.csv")

LABEL = "true_label"
MODELS = {"XGBoost": "xgb_proba", "ClinicalModernBERT": "bert_proba", "LateFusion": "combined_proba"}

pd.Series({name: roc_auc_score(df[LABEL], df[col]) for name, col in MODELS.items()}, name="AUROC").round(3)

XGBoost               0.848
ClinicalModernBERT    0.855
LateFusion            0.862
Name: AUROC, dtype: float64

In [9]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

y = df[LABEL]
pd.DataFrame({
    name: {
        "AUROC": roc_auc_score(y, df[col]),
        "AUPRC": average_precision_score(y, df[col]),
        "Brier": brier_score_loss(y, df[col]),
    }
    for name, col in MODELS.items()
}).T.round(3)

,AUROC,AUPRC,Brier
XGBoost,0.848,0.226,0.105
ClinicalModernBERT,0.855,0.212,0.092
LateFusion,0.862,0.256,0.090


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

INK, GREY, LIGHT = "#222222", "#7a7a7a", "#cfcfcf"
PAPER_STYLE = {
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8, "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.edgecolor": INK, "axes.linewidth": 0.6, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "pdf.fonttype": 42, "svg.fonttype": "none",
}

LABEL = "true_label"
MODELS = {  # name: (column, colour, linewidth); main model last = drawn on top
    "XGBoost":            ("xgb_proba",      "#d9734e", 1.1),
    "ClinicalModernBERT": ("bert_proba",     "#13917f", 1.1),
    "LateFusion":         ("combined_proba", "#2f5597", 1.8),
}
TICKS, TICK_LABELS = [0, 0.25, 0.5, 0.75, 1], ["0", "0.25", "0.50", "0.75", "1"]



def plot_roc_pr(df, path=None, figsize=(5.0, 2.75)):
    """(a) ROC and (b) precision-recall curves on the test set."""
    y = df[LABEL].to_numpy()
    prev = y.mean()
    ref = dict(color=LIGHT, lw=0.8, ls=(0, (2, 2)), zorder=1)

    with plt.rc_context(PAPER_STYLE):
        fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=figsize)
        ax_roc.plot([0, 1], [0, 1], **ref)
        ax_pr.plot([0, 1], [prev, prev], **ref)
        ax_pr.text(0.01, prev + 0.015, f"Prevalence ({100*prev:.1f}%)", fontsize=6.5, color=GREY, va="bottom")

        for name, (col, colour, lw) in MODELS.items():
            s = df[col].to_numpy()
            fpr, tpr, _ = roc_curve(y, s)
            prec, rec, _ = precision_recall_curve(y, s)
            ax_roc.plot(fpr, tpr, color=colour, lw=lw, zorder=3, label=name)
            ax_pr.plot(rec, prec, color=colour, lw=lw, zorder=3)
            print(f"{name:<20} AUROC {roc_auc_score(y, s):.3f}  AUPRC {average_precision_score(y, s):.3f}")

        for ax, xl, yl in [(ax_roc, "False positive rate", "True positive rate"), (ax_pr, "Recall", "Precision")]:
            ax.set(xlim=(0, 1), ylim=(0, 1), xlabel=xl, ylabel=yl, aspect="equal")
            ax.set_xticks(TICKS, TICK_LABELS)
            ax.set_yticks(TICKS, TICK_LABELS)

        for ax, letter in zip((ax_roc, ax_pr), "ab"):
            ax.text(-0.3, 1.05, letter, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")

        h, l = ax_roc.get_legend_handles_labels()
        fig.legend(h[::-1], l[::-1], loc="lower center", ncol=len(MODELS), fontsize=7.5,
                   handlelength=2, columnspacing=2.2, bbox_to_anchor=(0.5, -0.02))
        fig.tight_layout(w_pad=2.5, rect=(0, 0.08, 1, 1))
        if path:
            fig.savefig(f"{path}.pdf", bbox_inches="tight")
            fig.savefig(f"{path}.png", dpi=800, bbox_inches="tight")
    return fig


df = pd.read_csv("test_predictions_before_calibration.csv")   # adjust path
fig = plot_roc_pr(df, path="roc_pr_after")

XGBoost              AUROC 0.848  AUPRC 0.226
ClinicalModernBERT   AUROC 0.855  AUPRC 0.212
LateFusion           AUROC 0.862  AUPRC 0.256


## 2. Risk stratification

### 2.1 General

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

INK, GREY, LIGHT = "#222222", "#7a7a7a", "#cfcfcf"
PAPER_STYLE = {
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8, "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.edgecolor": INK, "axes.linewidth": 0.6, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "pdf.fonttype": 42, "svg.fonttype": "none",
}

LABEL, PROBA = "true_label", "combined_proba"          # calibrated LateFusion predictions
CUTS = [0, 0.02, 0.05, 0.10, 1.0001]
GROUPS = ["Low", "Moderate", "High", "Very high"]
RANGES = ["<2%", "2–5%", "5–10%", "≥10%"]
RAMP = ["#c9d6ec", "#8fa9d6", "#5578b8", "#2f5597"]     # one hue, light -> dark


def risk_table(df):
    g = pd.cut(df[PROBA], CUTS, right=False, labels=GROUPS)
    t = df.groupby(g, observed=False)[LABEL].agg(n="size", events="sum").reindex(GROUPS)
    t["rate"] = t.events / t.n
    t["lo"], t["hi"] = proportion_confint(t.events, t.n, alpha=0.05, method="wilson")
    prev = df[LABEL].mean()
    t["rr"] = t.rate / prev
    t["share_trials"] = t.n / t.n.sum()
    t["share_events"] = t.events / t.events.sum()
    return t, prev


def plot_risk_groups(df, path=None, figsize=(6.6, 2.8)):
    t, prev = risk_table(df)
    with plt.rc_context(PAPER_STYLE):
        fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=figsize, gridspec_kw=dict(width_ratios=[1.15, 1]))

        # (a) observed event rate per risk group
        x = np.arange(len(GROUPS))
        ax_a.bar(x, 100 * t.rate, width=0.62, color=RAMP, zorder=2)
        ax_a.errorbar(x, 100 * t.rate, yerr=[100 * (t.rate - t.lo), 100 * (t.hi - t.rate)],
                      fmt="none", ecolor=INK, elinewidth=0.7, capsize=2, capthick=0.7, zorder=3)
        ax_a.axhline(100 * prev, color=GREY, lw=0.8, ls=(0, (2, 2)), zorder=1)
        ax_a.annotate(f"Overall\nprevalence\n({100*prev:.1f}%)", xy=(1.0, 100 * prev), xycoords=("axes fraction", "data"),
                      xytext=(3, 0), textcoords="offset points", fontsize=6.5, color=GREY, va="center",
                      linespacing=1.1, annotation_clip=False)
        for xi, (_, r) in zip(x, t.iterrows()):
            ax_a.text(xi, 100 * r.hi + 0.6, f"{100*r.rate:.1f}%\n×{r.rr:.2f}", ha="center",
                      va="bottom", fontsize=6.5, linespacing=1.15)
        ax_a.set_xticks(x, [f"{g}\n{rg}" for g, rg in zip(GROUPS, RANGES)])
        ax_a.tick_params(axis="x", length=0)
        ax_a.set_ylabel("Observed positive trials (%)")
        ax_a.set_ylim(0, 100 * t.hi.max() * 1.3)
        ax_a.set_xlabel("Risk group (predicted probability)")

        # (b) share of trials vs share of positive trials
        rows = [("Positive\ntrials", t.share_events), ("All trials", t.share_trials)]
        for yi, (lab, shares) in enumerate(rows):
            left = 0
            for k, s in enumerate(shares):
                ax_b.barh(yi, 100 * s, left=left, height=0.55, color=RAMP[k],
                          edgecolor="white", linewidth=1, zorder=2)
                if s > 0.06:
                    ax_b.text(left + 50 * s, yi, f"{100*s:.0f}%", ha="center", va="center",
                              fontsize=6.5, color="white" if k >= 2 else INK)
                left += 100 * s
        ax_b.set_yticks([0, 1], [r[0] for r in rows])
        ax_b.tick_params(axis="y", length=0)
        ax_b.set_xlim(0, 100)
        ax_b.set_xticks([0, 25, 50, 75, 100])
        ax_b.set_xlabel("Proportion (%)")
        ax_b.spines["left"].set_visible(False)
        ax_b.set_ylim(-0.6, 1.9)
        handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in RAMP]
        ax_b.legend(handles, GROUPS, loc="upper center", ncol=4, fontsize=6.5, handlelength=1,
                    handleheight=1, columnspacing=1.2, bbox_to_anchor=(0.5, 1.02))

        for ax, letter in [(ax_a, "a"), (ax_b, "b")]:
            ax.text(-0.18, 1.04, letter, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")

        fig.tight_layout(w_pad=4)
        if path:
            fig.savefig(f"{path}.pdf", bbox_inches="tight")
            fig.savefig(f"{path}.png", dpi=800, bbox_inches="tight")
    return fig, t


# --- notebook usage ---------------------------------------------------------------------------
df = pd.read_csv("test_predictions_after_calibration.csv")   # adjust path
fig, table = plot_risk_groups(df, path="risk_groups")
table.round(3)

,n,events,rate,lo,hi,rr,share_trials,share_events
combined_proba,,,,,,,,
Low,3547,22,0.006,0.004,0.009,0.126,0.561,0.071
Moderate,948,26,0.027,0.019,0.040,0.559,0.150,0.084
High,738,58,0.079,0.061,0.100,1.602,0.117,0.187
Very high,1085,204,0.188,0.166,0.212,3.832,0.172,0.658


### 2.2 By Phase

In [19]:
import os
from datasets import load_from_disk
from aidose import END_POINT_HF_DATASET_PATH

def highest_phase(x):
    vals = [int(v) for v in np.atleast_1d(x) if v is not None and not (isinstance(v, float) and np.isnan(v))]
    return max(vals) if vals else np.nan

def map_stage(p):
    if pd.isna(p):
        return "Unknown"
    return {1: "Early", 2: "Early", 3: "Mid", 4: "Late", 5: "Late"}.get(int(p), "Unknown")

test_set = load_from_disk(os.path.join(END_POINT_HF_DATASET_PATH, "test")).to_pandas()
df = pd.read_csv("test_predictions_after_calibration.csv").merge(
    test_set[["METADATA_nctId", "FEATURE_phases"]], left_on="nctid", right_on="METADATA_nctId", how="inner")
df["stage"] = df["FEATURE_phases"].apply(highest_phase).apply(map_stage)
df["stage"].value_counts()

stage
Mid        3057
Late       2332
Early       806
Unknown     123
Name: count, dtype: int64

In [23]:
STAGE_COL = "stage"                      # adjust to your column name
STAGES = ["Early", "Mid", "Late"]


def plot_risk_by_stage(df, path=None, figsize=(7.0, 2.6)):
    """Observed proportion of positive trials per risk group, within each development stage."""
    tables = {s: risk_table(df[df[STAGE_COL] == s]) for s in STAGES}
    ymax = 100 * max(t.hi.max() for t, _ in tables.values()) * 1.25
    x = np.arange(len(GROUPS))

    with plt.rc_context(PAPER_STYLE):
        fig, axes = plt.subplots(1, len(STAGES), figsize=figsize, sharey=True)
        for ax, (stage, (t, prev)) in zip(axes, tables.items()):
            ax.bar(x, 100 * t.rate, width=0.62, color=RAMP, zorder=2)
            ax.errorbar(x, 100 * t.rate, yerr=[100 * (t.rate - t.lo), 100 * (t.hi - t.rate)],
                        fmt="none", ecolor=INK, elinewidth=0.7, capsize=2, capthick=0.7, zorder=3)
            ax.axhline(100 * prev, color=GREY, lw=0.8, ls=(0, (2, 2)), zorder=1)
            for xi, (_, r) in zip(x, t.iterrows()):
                ax.text(xi, 100 * r.hi + 0.8, f"{100*r.rate:.1f}%", ha="center", va="bottom", fontsize=6.5)
            ax.set_xticks(x, ["Low", "Mod.", "High", "Very\nhigh"])
            ax.tick_params(axis="x", length=0)
            ax.set_ylim(0, ymax)
            ax.set_title(stage, fontsize=8, loc="left", pad=14)
            ax.text(0, 1.03, f"n = {int(t.n.sum())} · prevalence {100*prev:.1f}%", transform=ax.transAxes,
                    fontsize=6.5, color=GREY, va="bottom")
        axes[0].set_ylabel("Observed positive trials (%)")
        fig.supxlabel("Risk group", fontsize=8, y=0.04)
        fig.tight_layout(w_pad=1.5)
        if path:
            fig.savefig(f"{path}.pdf", bbox_inches="tight")
            fig.savefig(f"{path}.png", dpi=800, bbox_inches="tight")
    return fig, tables


# --- notebook usage ---------------------------------------------------------------------------
fig, stage_tables = plot_risk_by_stage(df, path="risk_groups_by_stage")